# Notebook de entrenamiento de modelos

En este Notebook, se toman los datos crudos del dataset de Tipo de Cubierta Forestal de la base da datos mysql alojados en la tabla **forest_cover_processed** para entrenar los siguientes modelos:

- Support Vector Machine
- Decision Tree
- K-Nearest Neighbours

Estos modelos se usan para predecir a el tipo de cobertura forestal en base a variables cartográficas.

### Preprocesamiento de datos

En la siguiente celda, se define la función que preprocesa los datos para pasarlos por los modelos

In [21]:
# ========= Preprocesamiento de datos para el conjunto de datos de pingüinos =========

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix


def preprocess_data(df):
    """

    df = df.copy()

    target = "Cover_Type"

    cat_cols = ["Soil_Type", "Wilderness_Area"]

    num_cols = [
        "Elevation",
        "Aspect",
        "Slope",
        "Horizontal_Distance_To_Hydrology",
        "Vertical_Distance_To_Hydrology",
        "Horizontal_Distance_To_Roadways",
        "Hillshade_9am",
        "Hillshade_Noon",
        "Hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points"
    ]


    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[cat_cols] = df[cat_cols].fillna("Unknown")

    # OneHotEncoder para variables categóricas
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    df_cat = ohe.fit_transform(df[cat_cols])
    cat_feature_names = ohe.get_feature_names_out(cat_cols)
    df_cat_df = pd.DataFrame(df_cat, columns=cat_feature_names, index=df.index)

    # Concatenar numéricas y categóricas
    df_processed = pd.concat([df[num_cols], df_cat_df, df[target]], axis=1)
    """

    target = "Cover_Type"

    #X = df_processed.drop(columns=target)
    X = df.drop(columns=target)
    
    y = df[target]

    #encoders = {"onehot": ohe}

    #return df_processed, X, y, encoders
    return df, X, y, #encoders



In [10]:
df = pd.read_csv("dataset_test\dataset_test.csv")

In [4]:
df.columns

Index(['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology',
       'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways',
       'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
       'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1', 'Soil_Type1',
       'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 'Soil_Type5', 'Soil_Type6',
       'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 'Soil_Type10', 'Soil_Type11',
       'Soil_Type12', 'Soil_Type13', 'Soil_Type14', 'Soil_Type15',
       'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19',
       'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23',
       'Soil_Type24', 'Soil_Type25', 'Soil_Type26', 'Soil_Type27',
       'Soil_Type28', 'Soil_Type29', 'Soil_Type30', 'Soil_Type31',
       'Soil_Type32', 'Soil_Type33', 'Soil_Type34', 'Soil_Type35',
       'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39',
       'Soil_Type40', 'Wilderness_Area2', 'Wilderness_Area3',
       'Wilderness_Area4

### Definición de funciones para conexión a la BD

En la siguiente celda, se definen las funciones que perimten hacer la conexión con la base de datos, así como escribir y cargar datos a la misma.

In [ ]:
import mysql.connector
import pandas as pd
import os
import time
from sqlalchemy import create_engine


def get_engine():
    """Crea el engine de SQLAlchemy (USA ESTO en lugar del string URI)"""
    url = (
        f"mysql+pymysql://{MYSQL_CONFIG['user']}:"
        f"{MYSQL_CONFIG['password']}@"
        f"{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/"
        f"{MYSQL_CONFIG['database']}"
    )
    return create_engine(url)


def wait_for_db(retries=10, sleep=3):
    """Espera a que la DB esté lista"""
    engine = get_engine()
    for i in range(retries):
        try:
            with engine.connect():
                print("✅ DB ready")
                return
        except Exception as e:
            if i == retries - 1:
                raise RuntimeError(f"Database not reachable: {e}")
            print(f"⏳ Waiting for DB... ({i+1}/{retries})")
            time.sleep(sleep)


def load_penguins(df):
    """Carga los datos del CSV a MySQL"""
    # Esperar a que la DB esté lista

    cat_cols = ["Soil_Type", "Wilderness_Area"]

    num_cols = [
        "Elevation",
        "Aspect",
        "Slope",
        "Horizontal_Distance_To_Hydrology",
        "Vertical_Distance_To_Hydrology",
        "Horizontal_Distance_To_Roadways",
        "Hillshade_9am",
        "Hillshade_Noon",
        "Hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points"
    ]

    expected_cols = cat_cols + num_cols + ["Cover_Type"]
    
    if not all(col in df.columns for col in expected_cols):
        raise ValueError(f"El CSV debe contener las columnas: {expected_cols}")
    
    # 🔑 SOLUCIÓN: Usar el engine en lugar del string URI
    engine = get_engine()
    df[expected_cols].to_sql(
        "penguins_raw",
        con=engine,  # ← Usar engine aquí
        if_exists="append",
        index=False,
        method="multi"
    )
    print(f"✅ Loaded {len(df)} rows to penguins_raw")

## Entrenamiento de modelos

A continaución se definen las funciones para entrenar los modelos de ML.

In [20]:
# ========================= Funciones de entrenamiento para modelos de clasificación =========================

def train_decision_tree(df):
    """Entrena un modelo de Árbol de Decisión usando OneHotEncoder.
    
    Preprocesa los datos, divide en conjuntos de entrenamiento y prueba,
    entrena un DecisionTreeClassifier y evalúa su desempeño.
    
    Args:
        df (pd.DataFrame): DataFrame con los datos crudos de pingüinos.
        
    Returns:
        DecisionTreeClassifier: Modelo entrenado de Árbol de Decisión.
        
    Note:
        - Utiliza max_depth=5 para evitar overfitting
        - Aplica class_weight='balanced' para manejar desbalance de clases
        - Guarda los resultados en 'models_performance/decision_tree_results.txt'
        - Guarda el modelo en 'models/decision_tree.pkl'
    """
    #X, y, encoders = preprocess_data(df)
    df, X, y = preprocess_data(df)
    

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    model = DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    return evaluate_model(
        model, X_test, y_test,
        scaler=None,
        model_name="decision_tree"
    )


def train_svm(df):
    """Entrena un modelo de SVM usando OneHotEncoder.
    
    Preprocesa los datos, aplica normalización estándar, divide en conjuntos
    de entrenamiento y prueba, entrena un SVC y evalúa su desempeño.
    
    Args:
        df (pd.DataFrame): DataFrame con los datos crudos de pingüinos.
        
    Returns:
        SVC: Modelo entrenado de SVM.
        
    Note:
        - Utiliza kernel='rbf' para capturar relaciones no lineales
        - Aplica StandardScaler para normalizar las características
        - Aplica class_weight='balanced' para manejar desbalance de clases
        - Guarda los resultados en 'models_performance/svm_results.txt'
        - Guarda el modelo y el scaler en 'models/svm.pkl'
    """
    X, y, encoders = preprocess_data(df)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    model = SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    )

    model.fit(X_train, y_train)

    return evaluate_model(
        model, X_test, y_test,
        scaler=scaler,
        model_name="svm"
    )

def train_knn(df):
    """Entrena un modelo de KNN usando OneHotEncoder.
    
    Preprocesa los datos, aplica normalización estándar, divide en conjuntos
    de entrenamiento y prueba, entrena un KNeighborsClassifier y evalúa su desempeño.
    
    Args:
        df (pd.DataFrame): DataFrame con los datos crudos de pingüinos.
        
    Returns:
        KNeighborsClassifier: Modelo entrenado de KNN.
        
    Note:
        - Utiliza n_neighbors=5 para determinar la clasificación
        - Aplica StandardScaler para normalizar las características
        - Guarda los resultados en 'models_performance/knn_results.txt'
        - Guarda el modelo y el scaler en 'models/knn.pkl'
    """
    X, y, encoders = preprocess_data(df)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    model = KNeighborsClassifier(n_neighbors=5)

    model.fit(X_train, y_train)

    return evaluate_model(
        model, X_test, y_test,
        scaler=scaler,
        model_name="knn"
    )

## Para evaluar los modelos

In [52]:
# ========================= Función de evaluación y persistencia de modelos =========================

def evaluate_model(model, X_test, y_test, scaler, model_name):
    """Evalúa el desempeño de un modelo entrenado.
    
    Genera predicciones, calcula métricas de evaluación (reporte de clasificación
    y matriz de confusión), guarda los resultados en un archivo de texto y
    persiste el modelo junto con sus componentes.
    
    Args:
        model: Modelo entrenado (DecisionTreeClassifier, SVC o KNeighborsClassifier).
        X_test (array-like): Características del conjunto de prueba.
        y_test (array-like): Etiquetas verdaderas del conjunto de prueba.
        encoders (dict): Diccionario con los OneHotEncoders para variables categóricas.
        scaler: StandardScaler utilizado para normalizar X (None si no se usó).
        model_name (str): Nombre del modelo (ej: 'decision_tree', 'svm', 'knn').
        
    Returns:
        model: El mismo modelo pasado como argumento.
        
    Note:
        - Genera un reporte de clasificación con precisión, recall y f1-score
        - Genera una matriz de confusión
        - Guarda los resultados en 'models_performance/{model_name}_results.txt'
        - Persiste el modelo en 'models/{model_name}.pkl'
    """
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)

    results_text = f"""
MODEL: {model_name.upper()}

CLASSIFICATION REPORT
{report}

CONFUSION MATRIX
{matrix}
"""

    with open(f"../minio/models_performance/{model_name}_results.txt", "w") as f:
        f.write(results_text)

    #save_model(model, encoders, scaler, f"minio/models/{model_name}.pkl")
    save_model(model, scaler, f"../minio/models/{model_name}.pkl")

    return model



## Para guardar/cargar modelos

In [57]:

# ========================= Funciones de persistencia de modelos =========================

import os
import pickle
import tempfile


def save_model(model, scaler, filename):
    """Guarda un modelo entrenado junto con sus componentes de preprocesamiento de forma atómica.
    
    Serializa el modelo, los encoders y el scaler en un archivo pickle para
    posterior utilización en predicciones. La escritura se realiza primero en
    un archivo temporal y luego se reemplaza el archivo final de manera atómica,
    evitando que otros procesos (por ejemplo una API en producción) lean un
    archivo incompleto durante el guardado.
    
    Args:
        model: Modelo entrenado (DecisionTreeClassifier, SVC o KNeighborsClassifier).
        encoders (dict): Diccionario con los OneHotEncoders para variables categóricas.
        scaler: StandardScaler utilizado para normalizar características (puede ser None).
        filename (str): Ruta y nombre del archivo donde guardar el modelo.
        
    Returns:
        None
        
    Note:
        - Guarda un diccionario con tres claves: 'model', 'encoders' y 'scaler'
        - Utiliza el protocolo binario de pickle
        - Si el archivo existe se sobrescribe de forma segura (atómica)
        - Evita condiciones de carrera cuando múltiples procesos leen/escriben modelos
        - El cambio de archivo actualiza el timestamp, permitiendo mecanismos de hot-reload
        
    Raises:
        OSError: Si ocurre un error durante la escritura o reemplazo del archivo.
        
    Example:
        >>> save_model(model, encoders, scaler, 'models/my_model.pkl')
    """

    # Asegurar que el directorio exista
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    payload = {
        "model": model,
        #"encoders": encoders,
        "scaler": scaler
    }

    # Escribir primero en archivo temporal en el mismo directorio
    dir_name = os.path.dirname(filename)

    with tempfile.NamedTemporaryFile(delete=False, dir=dir_name, suffix=".tmp") as tmp:
        pickle.dump(payload, tmp)
        tmp_path = tmp.name

    # Reemplazo atómico del archivo final
    os.replace(tmp_path, filename)


def load_model(filename):
    """Carga un modelo entrenado junto con sus componentes de preprocesamiento.
    
    Deserializa el modelo, los encoders y el scaler desde un archivo pickle
    previamente guardado.
    
    Args:
        filename (str): Ruta y nombre del archivo desde donde cargar el modelo.
        
    Returns:
        tuple: Tupla contiendo:
            - model: Modelo entrenado (DecisionTreeClassifier, SVC o KNeighborsClassifier).
            - encoders (dict): Diccionario con los OneHotEncoders para variables categóricas.
            - scaler: StandardScaler utilizado para normalizar características (puede ser None).
            
    Raises:
        FileNotFoundError: Si el archivo especificado no existe.
        pickle.UnpicklingError: Si el archivo no contiene un objeto pickle válido.
        
    Note:
        - El archivo debe haber sido creado con la función save_model()
        - Los componentes devueltos se pueden usar para realizar predicciones en nuevos datos
    
    Example:
        >>> model, encoders, scaler = load_model('models/my_model.pkl')
    """
    with open(filename, "rb") as f:
        data = pickle.load(f)

    return data["model"], data["encoders"], data["scaler"]


## Entrenar y guardar modelos

In [59]:
# ========================= Ejecución del script de entrenamiento =========================

if __name__ == "__main__":

    df = pd.read_csv("dataset_test\dataset_test.csv")

    print("Training Decision Tree...")
    train_decision_tree(df)

    print("Training SVM...")
    train_svm(df)

    print("Training KNN...")
    train_knn(df)

    print("All models trained and saved successfully.")

Training Decision Tree...
Training SVM...


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [6]:
# Buena ñero

In [49]:
results_path = os.path.relpath("minio/models_performance/decision_tree_results.txt")
print(results_path)

minio\models_performance\decision_tree_results.txt
